# Silver Layer: Claims, Policies, and Accidents

This notebook transforms Bronze insurance tables into cleaned Silver Delta tables.

Each dataset follows the same pattern:

1. Read the Bronze table.
2. Standardize names and data types.
3. Add dataset-specific derived fields.
4. Apply data-quality filters.
5. Write the Silver Delta table.

## Setup

Shared imports, schema loading, column-standardization helpers, validation helpers, and Delta write helpers.

In [0]:
import json
from functools import reduce

from pyspark.sql import functions as F

In [0]:
SCHEMA_PATH = "/Workspace/Users/wangwangming1975@gmail.com/insurance-claims/data/schemas"

In [0]:
def load_schema(file_name):
    with open(f"{SCHEMA_PATH}/{file_name}", "r", encoding="utf-8") as schema_file:
        return json.load(schema_file)


def get_column_case_insensitive(df, expected_name):
    matches = [column for column in df.columns if column.lower() == expected_name.lower()]
    if not matches:
        raise ValueError(f"Column '{expected_name}' was not found. Available columns: {df.columns}")
    return matches[0]


def rename_from_schema(df, schema, use_alias=True):
    for spec in schema:
        source_col = spec["name"]
        target_col = spec.get("alias", source_col) if use_alias else source_col
        actual_col = get_column_case_insensitive(df, source_col)

        if actual_col != target_col:
            df = df.withColumnRenamed(actual_col, target_col)

    return df


def require_non_null(df, columns):
    required_condition = reduce(
        lambda condition, column: condition & F.col(column).isNotNull(),
        columns,
        F.lit(True),
    )
    return df.filter(required_condition)


def write_silver_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(table_name)
    )

## 1. Claims

Clean claim records by flattening nested Bronze fields, standardizing dates, creating analysis flags, and writing `silver_claims`.

### Read Bronze Claims

Read the raw claims source table.

In [0]:
claims_df = spark.table("bronze_claims")

### Transform Claims

Flatten nested claim, collision, driver, and incident structures. Standardize identifiers during selection so a separate rename step is not needed.

In [0]:
silver_claims_df = claims_df.select(
    F.col("claim_no").alias("claim_number"),
    F.col("policy_no").alias("policy_number"),
    F.col("claim_datetime"),

    F.col("claim_amount.injury").alias("injury_claim_amount"),
    F.col("claim_amount.property").alias("property_claim_amount"),
    F.col("claim_amount.vehicle").alias("vehicle_claim_amount"),
    F.col("claim_amount.total").alias("total_claim_amount"),

    F.col("collision.number_of_vehicles_involved"),
    F.col("collision.type").alias("collision_type"),

    F.col("driver.age").alias("driver_age"),
    F.col("driver.insured_relationship"),
    F.col("driver.license_issue_date"),

    F.col("incident.date").alias("incident_date"),
    F.col("incident.hour").alias("incident_hour"),
    F.col("incident.severity"),
    F.col("incident.type").alias("incident_type"),

    F.col("months_as_customer"),
    F.col("number_of_witnesses"),
    F.col("suspicious_activity"),
    F.col("ingestion_timestamp"),
)

### Standardize Claim Types

Convert date and timestamp columns to Spark types. Malformed license issue dates are returned as null.

In [0]:
silver_claims_df = (
    silver_claims_df
    .withColumn("claim_datetime", F.to_timestamp("claim_datetime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("incident_date", F.to_date("incident_date", "dd-MM-yyyy"))
    .withColumn(
        "license_issue_date",
        F.when(
            F.col("license_issue_date").rlike("^[0-9]{2}-[0-9]{2}-[0-9]{4}$"),
            F.to_date("license_issue_date", "dd-MM-yyyy"),
        ),
    )
)

### Create Claim Features

Add derived columns used by downstream claim analysis.

In [0]:
silver_claims_df = silver_claims_df.withColumn(
    "injury_to_person",
    F.when(F.col("injury_claim_amount") > 0, F.lit(1)).otherwise(F.lit(0)),
)

### Validate Claims

Keep records with required identifiers, a valid incident date, and an incident hour between 0 and 24.

In [0]:
silver_claims_df = require_non_null(
    silver_claims_df,
    ["claim_number", "policy_number", "incident_date"],
).filter(F.col("incident_hour").between(0, 24))

### Write Silver Claims

Overwrite `silver_claims` with the cleaned claims dataset.

In [0]:
write_silver_table(silver_claims_df, "silver_claims")

## 2. Policies

Clean policy records by applying schema aliases, parsing dates, validating required business fields, and writing `silver_policies`.

### Read Bronze Policies

Load the policy schema and the raw policies source table.

In [0]:
policies_schema = load_schema("sql/policies.json")
policies_df = spark.table("bronze_policies")

### Standardize Policy Fields

Apply schema aliases and convert `driver_dob` to a Spark date, returning null for malformed values.

In [0]:
policies_df = rename_from_schema(policies_df, policies_schema)
policies_df = policies_df.withColumn(
    "driver_dob",
    F.expr("try_to_date(driver_dob, 'dd-MM-yyyy')"),
)

### Validate Policies

Keep policies with non-null customer, policy, issue-date, premium, and insured-sum fields.

In [0]:
policies_df = require_non_null(
    policies_df,
    ["customer_id", "policy_number", "issue_date", "premium", "sum_insured"],
)

### Write Silver Policies

Overwrite `silver_policies` with the validated policy dataset.

In [0]:
write_silver_table(policies_df, "silver_policies")

In [0]:
spark.table("silver_policies").count()

## 3. Accidents

Clean accident records by standardizing schema column names, casting numeric fields, validating required location fields, and writing `silver_accidents`.

### Read Bronze Accidents

Load the accidents schema and the raw accidents source table.

In [0]:
accidents_schema = load_schema("s3/accidents.json")
accidents_df = spark.table("bronze_accidents")

### Standardize Accident Fields

Apply schema column names and cast date, ZIP code, injury, fatality, and collision identifier fields.

In [0]:
accidents_df = rename_from_schema(accidents_df, accidents_schema, use_alias=False)

accidents_df = (
    accidents_df
    .withColumn("accident_date", F.to_timestamp("accident_date"))
    .withColumn("zip_code", F.expr("try_cast(zip_code as double)").cast("int"))
    .withColumn("number_of_persons_injured", F.expr("try_cast(number_of_persons_injured as int)"))
    .withColumn("number_of_persons_killed", F.expr("try_cast(number_of_persons_killed as int)"))
    .withColumn("number_of_motorist_injured", F.expr("try_cast(number_of_motorist_injured as int)"))
    .withColumn("number_of_motorist_killed", F.expr("try_cast(number_of_motorist_killed as int)"))
    .withColumn("collision_id", F.col("collision_id").cast("string"))
)

### Validate Accidents

Keep records with a valid accident timestamp, collision identifier, latitude, and longitude.

In [0]:
accidents_df = require_non_null(
    accidents_df,
    ["accident_date", "collision_id", "latitude", "longitude"],
)

### Write Silver Accidents

Overwrite `silver_accidents` with the cleaned accidents dataset.

In [0]:
write_silver_table(accidents_df, "silver_accidents")

In [0]:
spark.table("silver_accidents").count()